# regrets-to-inform: A Data Science Skills Showcase
## Hypothesis: 
The South African Jobs market is broken for graduates. Jobs are mostly unavailable, hiring managers ghost applicants, and many jobs were never available at the time you clicked apply. Companies leave listings open for roles that are already filled, or in some extreme cases, appear to be engaging in data harvesting for ATS training. 

In [126]:
import numpy as np
from linkedin_parser import parse_jobs
import pandas as pd
import polars as pl
import seaborn as sns 
import json
import glob
from rapidfuzz import fuzz, utils
import uuid
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 7)
pd.set_option('display.precision', 8)
import re
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
import ibis
import pyarrow
con = ibis.polars.connect()
ibis.options.interactive = True
warnings.filterwarnings('ignore')
from ibis import _
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
FIG_W = 14

Let us set up the file processing pipeline

In [127]:
file_path1 = 'nonfunctionapps/unprocessed_email_dump/unprocessed'
file_path2 = 'nonfunctionapps/email_dump/processed'
EMPTY_LIST = ['Unknown', 'None', None, 'nan', 'none', 'NA', 'NaN', ""]

file_list = glob.glob(f'{file_path1}/*.json')
obj_list = []
for files in file_list:
    with open(files, 'r') as f:
        temp = json.load(f)
    obj_list.append(temp)
file_list = glob.glob(f'{file_path2}/*.json')
for files in file_list:
    with open(files, 'r') as f:
        temp = json.load(f)
    obj_list.append(temp)

Our first DataFrame: Emails

In [128]:
emails = pd.json_normalize(obj_list)
emails['record_type'] = 'email'
emails['record_type'] = emails['record_type'].astype('category')

In [129]:
#Some further df cleaning
FAMILIES = {
    'Data / Analytics':      r'\bdata anal|\banalytics\b|\bBI\b|business intel|insight',
    'Data Science / ML':     r'\bdata scien|machine learn|\bML\b|\bAI engineer|artificial intell',
    'Data Engineering':      r'\bdata engineer|\betl\b|databrick|\bspark\b|data warehouse|\bdbt\b',
    'Operations Research':   r'operations research|\bOR\b|optimis|optimiz|supply chain|logistics',
    'Finance / Quant':       r'financ|quant|actuar|\brisk\b|econom',
    'Consulting / Strategy': r'consult|strateg|advisor',
    'Software / Dev':        r'\bsoftware\b|\bbackend\b|\bfrontend\b|full.?stack|\bdev\b',
    'Project / Product':     r'project manag|product manag|scrum|agile|\bPMO\b',
}
emails.columns = ['filename', # string
                'to', # string, email
                'from', # string, email 
                'subject', # string
                'cc', # string(s)
                'date', # ISO 8601 date
                'body', # string
                'company', # string
                'portal', # string (category)
                'notification_type', #string (category)
                'role', # string
                'update_category',# string (category)
                'is_automated', # boolean
                'urgency', # string (category)
                'interview_type', # string (category)
                'interview_platform', # string (category)
                'test_platform', # string (category)
                'test_duration_mins', # int
                'action_required', # boolean
                'action_type', # string (category)
                'summary', #string
                'interview_date', #ISO 8601 date
                'application_deadline', #ISO 8601 date
                'action_deadline',
                'work_mode', # string (category)
                'location', # string (category)
                'recruiter_name', # string
                'recruiter_email', # string
                'record_type'# string (category)
                ]
col_types = {'filename': str,
                'to': str,
                'from':'string',
                'subject':'string',
                'cc': str,
                'date': str,
                'body': str,
                'company': str,
                'portal': 'category',
                'notification_type':'category',
                'role':'string',
                'update_category':'category',
                'is_automated':bool,
                'urgency':'category',
                'interview_type':'category',
                'interview_platform':'category',
                'test_platform':'category',
                'test_duration_mins':int,
                'action_required':bool,
                'action_type':'category',
                'summary':'string',
                'interview_date':'string',
                'application_deadline':'string',
                'action_deadline':'string',
                'work_mode':'category',
                'location':'category',
                'recruiter_name':'string',
                'recruiter_email' :'string',
                'record_type':'category'
                }
timecols = ['date','interview_date','application_deadline','action_deadline']
emails = emails.astype(col_types, errors='ignore')
def normalize_company_and_role(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col("company")
        .str.to_lowercase() 
        # --- COMPANY NORMALIZATION ---
        # Remove punctuation
        .str.replace_all(r"[^\w\s]", " ")
        # Remove common corporate suffixes (using word boundaries \b)
        .str.replace_all(r"\b(inc|llc|corp|corporation|ltd|limited|co)\b", "")
        # Remove extra whitespaces
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
        .fill_null("unknown")
        .alias("company_clean"), # Saving as a new column so you don't lose original data

        # --- ROLE NORMALIZATION ---
        pl.col("role")
        .str.to_lowercase()
        # Remove anything inside brackets e.g. "(Remote)" or "[Contract]"
        .str.replace_all(r"\(.*?\)|\[.*?\]", "")
        # Remove punctuation
        .str.replace_all(r"[^\w\s]", " ")
        # Remove extra whitespaces
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
        .fill_null("unknown")
        .str.to_lowercase()
        .alias("role_clean")
    ])
def clean_and_prep(df: pl.DataFrame, date_col: str = None, date: str = None) -> pl.DataFrame:
    # 1. Strip timezones if the column exists
    if date:
        date_col = date
    if date_col in df.columns and df.columns.__contains__('from'):
        df = df.with_columns(pl.col(date_col).dt.replace_time_zone(None))
# 2. Clean and fill text columns
        return normalize_company_and_role(df.with_columns([
            # --- NEW CLEANING LOGIC ---
            # Fill nulls with 'from', then extract domain name, then strip punctuation/suffixes
            pl.col("company")
            .fill_null(pl.col("from"))  # Use from column if company is null
            .str.to_lowercase()
            # Clean email structure if present (e.g., "user@domain.com" -> "domain")
            .str.extract(r"@([^\.]+)")  # Keeps only the part after @ and before first dot
            .fill_null(pl.col("company").fill_null(pl.col("from")).str.to_lowercase()) # if not an email, revert to text
            # ---------------------------
            .str.replace_all(r"[^\w\s]", " ")
            .str.replace_all(r"\b(inc|llc|corp|corporation|ltd|limited|co|group)\b", "")
            .str.replace_all(r"\s+", " ").str.strip_chars()
            .alias("company_clean"),
            pl.col("role").fill_null("unknown").str.to_lowercase()
            .str.replace_all(r"\(.*?\)|\[.*?\]", "")
            .str.replace_all(r"[^\w\s]", " ")
            .str.replace_all(r"\s+", " ").str.strip_chars()
            .alias("role_clean")
        ]))
    elif date_col in df.columns:
        df = df.with_columns(pl.col(date_col).dt.replace_time_zone(None))

    elif df.columns.__contains__('from'):
        return normalize_company_and_role(df.with_columns([
            # --- NEW CLEANING LOGIC ---
            # Fill nulls with 'from', then extract domain name, then strip punctuation/suffixes
            pl.col("company")
            .fill_null(pl.col("from"))  # Use from column if company is null
            .str.to_lowercase()
            # Clean email structure if present (e.g., "user@domain.com" -> "domain")
            .str.extract(r"@([^\.]+)")  # Keeps only the part after @ and before first dot
            .fill_null(pl.col("company").fill_null(pl.col("from")).str.to_lowercase()) # if not an email, revert to text
            # ---------------------------
            .str.replace_all(r"[^\w\s]", " ")
            .str.replace_all(r"\b(inc|llc|corp|corporation|ltd|limited|co|group)\b", "")
            .str.replace_all(r"\s+", " ").str.strip_chars()
            .alias("company_clean")]))
    else: return normalize_company_and_role(df.with_columns([
            # --- NEW CLEANING LOGIC ---
            # Fill nulls with 'from', then extract domain name, then strip punctuation/suffixes
            pl.col("company")
            # Clean email structure if present (e.g., "user@domain.com" -> "domain")
            .str.extract(r"@([^\.]+)")  # Keeps only the part after @ and before first dot
            .fill_null(pl.col("company")) # if not an email, revert to text
            # ---------------------------
            .str.replace_all(r"[^\w\s]", " ")
            .str.replace_all(r"\b(inc|llc|corp|corporation|ltd|limited|co|group)\b", "")
            .str.replace_all(r"\s+", " ").str.strip_chars()
            .alias("company_clean")]))

In [130]:
# Cleaning the dates
emails[timecols] = emails[timecols].apply(
    pd.to_datetime, format='mixed', errors='coerce', utc=True, cache=True
)

In [131]:
#drop the non-work related emails:
emails = emails[~emails['notification_type'].isin(['Unrelated', 'Job ad'])]
EMPTY_LIST = ['Unknown', 'None', None, 'nan', 'none', 'NA', 'NaN', ""]
#emails = emails[~emails['body'].isin(EMPTY_LIST)]
all_fields_empty = (
    emails['notification_type'].isin(['Unknown']) & 
    emails['company'].isin(EMPTY_LIST) & 
    emails['role'].isin(EMPTY_LIST) &
    ~emails['is_automated'] & 
    emails['portal'].isin(EMPTY_LIST) &
    emails['action_type'].isin(EMPTY_LIST)
)
emails = emails[~all_fields_empty]
emails.reset_index(drop=True, inplace=True)

In [132]:
# LinkedIn Quick Apply data
lnkdin = pd.read_csv('Linkedin_QuickApply.csv', index_col=0)
# Preserve employer screening questions before dropping feature columns
questions = lnkdin[[col for col in lnkdin.columns
                     if col.startswith('Ask_') or col.startswith('Req_')]].copy()
questions['job_url'] = lnkdin['Job_Url']
lnkdin.drop([col for col in lnkdin.columns
             if col.startswith('Ask_') or col.startswith('Req_')], axis=1, inplace=True)
lnkdin.columns = ['linkedin_date', 'company', 'role', 'job_url']
# LinkedIn exports dates in MM/DD/YYYY — dayfirst=False is correct.
# Using dayfirst=True previously swapped month/day for any day<=12,
# pushing April dates into future months (Oct, Aug, Nov, etc.).
lnkdin['linkedin_date'] = pd.to_datetime(lnkdin['linkedin_date'], dayfirst=False)

In [133]:
# Process emails to get jobs
emails.drop(['action_deadline','cc','recruiter_email', 'recruiter_name', 'location', 'work_mode', 'application_deadline', 'interview_date'], axis=1, inplace=True)
emails['uid'] = emails['filename'].str.split('.', expand=True)[0]
emails.drop(['filename', 'to', 'subject', 'body', 'test_platform', 'test_duration_mins', 'record_type'], inplace=True, axis=1)#Remove this at some point

In [134]:
emails = emails.reindex(columns=['uid', 'date', 'notification_type', 'update_category', 'company', 'role', 'urgency', 'action_required', 'action_type', 'is_automated', 'portal', 'from',  'interview_platform', 'interview_type', 'summary' ])
emails.sort_values('company', inplace=True)
emails.reset_index(drop=True, inplace=True)
emails['company']=emails['company'].replace("Unknown",None)
emails['date'] = emails['date'].apply(
    pd.to_datetime, format='mixed', errors='coerce', utc=True, cache=True
)
lnkdin['linkedin_date'] = lnkdin['linkedin_date'].apply(
    pd.to_datetime, format='mixed', errors='coerce', utc=True, cache=True
)


In [135]:
df = parse_jobs('nonfunctionapps/linkedindump.txt')
df['applied_date'] = df['applied_date'].apply(
    pd.to_datetime, format='mixed', errors='coerce', utc=True, cache=True
)
df.sort_values('company', inplace=True)
df.reset_index(drop=True, inplace=True)
emails.sort_values('company', inplace=True)
emails.reset_index(drop=True, inplace=True)
lnkdin.sort_values('company', inplace=True)
lnkdin.reset_index(drop=True, inplace=True)
df.to_csv('df.csv')
lnkdin.to_csv('lnkdin.csv')
emails.to_csv('emails.csv')

In [136]:
#converd dataframe to polars
df = pl.from_dataframe(df)
lnkdin = pl.from_dataframe(lnkdin)
emails=pl.from_dataframe(emails)
emails_clean = clean_and_prep(emails)
lnkdin_clean = clean_and_prep(lnkdin)
df_clean = clean_and_prep(df)



In [137]:



# Convert to Polars and clean
# emails_clean = clean_and_prep(emails, "date")
# lnkdin_clean = clean_and_prep(lnkdin, "linkedin_date")
# df_clean = clean_and_prep(df, "applied_date")

# Connect to Ibis
con = ibis.polars.connect({
    'mail': emails_clean, 
    'linkedin1': lnkdin_clean, 
    'linkedin2': df_clean
})
df


role,company,location,work_type,is_open,listing_status,raw_applied,applied_date,raw_posted,posted_date,raw_reposted,reposted_date,app_viewed,resume_downloaded,outcome
str,str,str,str,bool,str,str,"datetime[μs, UTC]",str,datetime[μs],str,datetime[μs],bool,bool,str
"""Data Engineer""","""4Sight Holdings Limited""","""Centurion""","""On-site""",false,"""closed""","""Applied 1mo ago""",2026-05-10 11:09:20.229809 UTC,null,null,null,null,false,false,"""Not moving forward"""
"""Industrial Engineer""","""4Sight OT Simulation""","""Centurion""","""Hybrid""",false,"""closed""","""Applied 1w ago""",2026-04-26 11:11:20.228070 UTC,null,null,null,null,false,false,null
"""Pricing and Systems Analyst - …","""A.P. Moller - Maersk""","""Johannesburg""",null,true,"""open""","""Applied just now""",2026-05-10 11:10:20.230428 UTC,"""Posted 1w ago""",2026-04-26 11:11:20.234629,null,null,false,false,null
"""Graduate in Training""","""ABB""","""Johannesburg""",null,false,"""closed""",null,null,null,null,null,null,false,false,null
"""Intercompany Analyst""","""ABinBev SAB""","""Sandton""",null,false,"""closed""",null,null,null,null,null,null,false,false,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Developer""",null,"""Pretoria""","""On-site""",false,"""closed""","""Applied 2w ago""",2026-04-19 11:11:20.229629 UTC,null,null,null,null,false,false,null
"""Researcher Space Science""",null,"""City of Johannesburg""","""On-site""",false,"""closed""","""Applied 1mo ago""",2026-05-10 11:09:20.230133 UTC,null,null,null,null,true,true,null
"""Fleet Controller""",null,"""Johannesburg Metropolitan Area""","""Hybrid""",false,"""closed""","""Applied 1mo ago""",2026-05-10 11:09:20.230271 UTC,null,null,null,null,false,false,null


In [138]:

# Lets start with the SQL proof of concept 
# con = ibis.polars.connect(tables={'mail':emails_clean, 'linkedin1': lnkdin_clean, 'linkedin2':df_clean})

mail= con.table('mail')
linkedin1=con.table('linkedin1')
linkedin2=con.table('linkedin2')
con.list_tables()
rej_mail_fetch= """SELECT *
            FROM mail
            WHERE update_category = 'Rejection'
            OR notification_type = 'Rejection'
            """
mail_fetch ="""SELECT role_clean, company_clean, date, update_category, notification_type, is_automated, portal
            FROM mail
            """
li1_fetch= """SELECT role_clean, company_clean, linkedin_date
            FROM linkedin1
            """
li2_fetch= """SELECT role_clean, company_clean, location, work_type, applied_date
            FROM linkedin2
            """
mail_fetch = con.sql(mail_fetch)
li1_fetch = con.sql(li1_fetch)
li2_fetch = con.sql(li2_fetch)



In [139]:

# # 1. Define the grouping columns
# group_cols = ["company", "role"]
# mail_fetch.group_by(group_cols)
# mail_fetch
#group_expr = """SELECT mail.company, mail.role, linkedin2.company, linkedin2.role, COUNT(DISTINCT mail.update_category) AS distinct_roles FROM mail, linkedin2 GROUP BY company, role ORDER BY distinct_roles DESC"""
# outer_join_expr = """WITH MailGroups AS (
#                         SELECT 
#                             company_clean, 
#                             role_clean, 
#                             COUNT(DISTINCT update_category) AS distinct_roles, 
#                             COUNT(*) AS mail_count
#                         FROM mail
#                         GROUP BY company_clean, role_clean
#                     ),
#                     LinkedInGroups AS (
#                         SELECT 
#                             company_clean, 
#                             role_clean, 
#                             COUNT(*) AS linkedin_count
#                         FROM linkedin2
#                         GROUP BY company_clean, role_clean
#                     )
#                     SELECT 
#                         company_clean, 
#                         role_clean, 
#                         COALESCE(distinct_roles, 0) AS distinct_roles, 
#                         COALESCE(mail_count, 0) AS mail_count, 
#                         COALESCE(linkedin_count, 0) AS linkedin_count
#                     FROM MailGroups
#                     FULL OUTER JOIN LinkedInGroups 
#                         USING (company_clean, role_clean)
#                     ORDER BY mail_count DESC
# """

# # Execute the SQL
# all_applications = con.sql(outer_join_expr)


In [140]:

# View the results
# df = all_applications.to_pandas()

# group_expr = con.sql(group_expr)
# num_unique_roles = group_expr.distinct_roles.sum()
# num_unique_roles.cast(int16)


In [141]:
outer_join_expr = """
WITH MailGroups AS (
    SELECT 
        mail.company_clean AS m_company, 
        mail.role_clean AS m_role, 
        COUNT(DISTINCT mail.update_category) AS distinct_roles, 
        COUNT(mail.company_clean) AS mail_count,
        MIN(mail.date) AS mail_date,
        
        -- Extract the raw milestones
        MAX(CASE WHEN mail.update_category IN ('Personality Test Request') THEN 1 ELSE 0 END) AS has_p_test,
        MAX(CASE WHEN mail.update_category IN ('Technical Test Request') THEN 1 ELSE 0 END) AS has_t_test,
        MAX(CASE WHEN (mail.update_category LIKE 'Interview Invitation' OR mail.notification_type LIKE 'Interview') AND mail.is_automated != 1 THEN 1 ELSE 0 END) AS has_interview,
        MAX(CASE WHEN mail.update_category = 'Rejection' OR mail.notification_type = 'Rejection' THEN 1 ELSE 0 END) AS is_rejected_mail,
        MAX(CASE WHEN mail.update_category IN ('Confirmation of application') THEN 1 ELSE 0 END) AS is_confirmation,
        MAX(CASE WHEN mail.is_automated = 1 OR mail.update_category LIKE 'Bot/Auto-Reply' THEN 1 ELSE 0 END) AS is_automated
    FROM mail
    GROUP BY mail.company_clean, mail.role_clean
),
LinkedInGroups AS (
    SELECT 
        linkedin2.company_clean AS l_company, 
        linkedin2.role_clean AS l_role, 
        COUNT(linkedin2.company_clean) AS linkedin_count,
        MIN(linkedin2.applied_date) AS linkedin_date,
        
        -- Check for explicit LinkedIn rejection
        MAX(CASE WHEN linkedin2.outcome = 'Not moving forward' THEN 1 ELSE 0 END) AS is_rejected_li
    FROM linkedin2
    GROUP BY linkedin2.company_clean, linkedin2.role_clean
)
SELECT 
    COALESCE(final_m.m_company, final_l.l_company) AS company_clean, 
    COALESCE(final_m.m_role, final_l.l_role) AS role_clean, 
    
    -- Consolidate the flags
    COALESCE(final_m.has_p_test, 0) AS has_p_test,
    COALESCE(final_m.has_t_test, 0) AS has_t_test,
    COALESCE(final_m.is_confirmation, 0) AS is_confirmation,
    COALESCE(final_m.is_automated, 0) AS is_automated,
    COALESCE(final_m.has_interview, 0) AS has_interview,
    GREATEST(COALESCE(final_m.is_rejected_mail, 0), COALESCE(final_l.is_rejected_li, 0)) AS is_rejected,
    
    COALESCE(final_m.distinct_roles, 0) AS distinct_roles, 
    COALESCE(final_m.mail_count, 0) AS mail_count, 
    COALESCE(final_l.linkedin_count, 0) AS linkedin_count,
    
    CASE
        WHEN final_m.mail_date IS NULL THEN final_l.linkedin_date
        WHEN final_l.linkedin_date IS NULL THEN final_m.mail_date
        WHEN final_m.mail_date < final_l.linkedin_date THEN final_m.mail_date
        ELSE final_l.linkedin_date
    END AS first_applied_date
    
FROM MailGroups final_m
FULL OUTER JOIN LinkedInGroups final_l 
    ON final_m.m_company = final_l.l_company 
    AND final_m.m_role = final_l.l_role
"""

# Execute the query
all_applications = con.sql(outer_join_expr)

# Execute the query
all_applications = con.sql(outer_join_expr)
all_applications

┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━┓
┃ company_clean          ┃ role_clean            ┃ has_p_test ┃ has_t_test ┃ … ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━┩
│ string                 │ string                │ int32      │ int32      │ … │
├────────────────────────┼───────────────────────┼────────────┼────────────┼───┤
│ roomraccoon hotel tech │ join our talent pool… │          0 │          0 │ … │
│ future fibres          │ graduate program      │          0 │          0 │ … │
│ discovery              │ hr data analyst for … │          0 │          0 │ … │
│ teelor                 │ data analyst          │          0 │          0 │ … │
│ takealot com           │ logistic internship   │          0 │          0 │ … │
│ gravity group          │ junior low code deve… │          0 │          0 │ … │
│ dataloop               │ data analytics consu… │          0 │          0 │ … │
│ springer nature group  │ associate publisher   │          0 │          0 │ … │
│ akqa                   │ data analyst          │          0 │          0 │ … │
│ data strategie         │ internship data anal… │          0 │          0 │ … │
│ …                      │ …                     │          … │          … │ … │
└────────────────────────┴───────────────────────┴────────────┴────────────┴───┘

In [160]:
import plotly.graph_objects as go
from datetime import datetime, timedelta

# 1. AGGRESSIVE CLEANING & VERIFICATION
# We only count an interview if it has a valid platform or type assigned
verified_funnel = all_applications.mutate(
    # Ensure we are checking lowercase strings for 'none'
    is_valid_interview = (
        (all_applications.has_interview == 1)
    ).cast('int8')
).group_by(["company_clean", "role_clean"]).aggregate(
    has_p_test = all_applications.has_p_test.max(),
    has_t_test = all_applications.has_t_test.max(),
    # Use the new verification flag
    has_interview = _.is_valid_interview.max(), 
    is_rejected = all_applications.is_rejected.max(),
    first_date = all_applications.first_applied_date.min()
)

# 2. CALCULATE VOLUMES
n_total = verified_funnel.count().to_pandas()
n_t_test  = verified_funnel.filter(verified_funnel.has_t_test == 1).count().to_pandas()
n_p_test  = verified_funnel.filter(verified_funnel.has_p_test == 1).count().to_pandas()
n_int   = verified_funnel.filter(verified_funnel.has_interview == 1).count().to_pandas()
n_rej   = verified_funnel.filter(verified_funnel.is_rejected == 1).count().to_pandas()

# Balanced Rejection Splits
rej_after_int = verified_funnel.filter((verified_funnel.has_interview == 1) & (verified_funnel.is_rejected == 1)).count().to_pandas()
rej_after_test = verified_funnel.filter(((verified_funnel.has_t_test == 1) | (verified_funnel.has_p_test == 1)) & (verified_funnel.has_interview == 0) & (verified_funnel.is_rejected == 1)).count().to_pandas()
rej_initial = n_rej - rej_after_int - rej_after_test

# Ghosted/Active Logic (3-week rule)
today_dt = datetime(2026, 5, 10)
ghost_cutoff = today_dt - timedelta(days=21)

# Line 36 should look like this:
n_gho = (
    verified_funnel
    .filter([
        verified_funnel.is_rejected == 0,
        # Cast the column to match the expected comparison unit
        verified_funnel.first_date.cast("timestamp") < ghost_cutoff
    ])
    .count()
    .execute()
)

n_act = n_total - n_rej - n_gho

# 3. THE SANKEY (Slate & Emerald Theme)
# Applied, Tests, Interviews, Rejected, Ghosted, Active
colors = ["#3399FF", "#FF66B2", "#66FF66","#FF3333", "#FF9933", "#00CCCC"]


fig = go.Figure(data=[go.Sankey(
    arrangement = "freeform",
    valueformat = "d",
    node = dict(
      pad = 150, thickness = 30,
      label = [
          f"Applied ({n_total})", #0
          f"Psychometric Tests ({n_p_test})", #1
          f"Technical Tests ({n_t_test})", #2
          f"Interviews or Screenings({n_int})", #3
          f"Rejections ({n_rej})", #4
          f"Ghosted (>4wks) ({n_gho})", #5
          f"Active (<4wks) ({n_act})" #6
      ],
     color = colors
    ),
    link = dict(
      source = [0, 0, 0, 0, 0, 1, 2, 2, 1, 2, 3 ], 
      target = [1, 2, 3, 5, 6, 4, 3, 4, 3, 3, 4 ], 
      value  = [n_p_test, n_t_test, n_int, n_gho, n_act, (n_p_test - n_t_test), (n_t_test),  n_int, n_gho, n_act],
      color = "rgba(204, 153, 255, 0.15)"
    )
)])

fig.update_layout(
    title_text="<b>JOB SEARCH FUNNEL (MARCH -- MAY)</b>",
    template="plotly_dark",
    font=dict(size=14, family="Helvetica"),
    height=800, width=1100
)

print(f"Verified Interviews: {n_int}")
fig.show()

Verified Interviews: 7


In [ ]:
breakpoint()

In [ ]:

# from datetime import datetime

# today = datetime(2026, 5, 4)

# # Step A: Deduplicate - Group strictly by Company (to avoid role-duplicate issues)
# # We cast to String to prevent categorical errors and kill 'None' values
# unique_stages = all_events.with_columns([
#     pl.col("stage").cast(pl.String),
#     pl.col("company_clean").cast(pl.String).fill_null("unknown").str.to_lowercase().str.strip_chars()
# ]).filter(
#     ~pl.col("company_clean").is_in(["", "none", "unknown", "null"])
# ).group_by(["company_clean", "role_clean", "stage"]).agg(
#     pl.col("event_date").min().alias("stage_date")
# )

# # Step B: Final Job Summary
# job_funnel = unique_stages.group_by(["company_clean", "role_clean"]).agg([
#     pl.col("stage_date").max().alias("last_update"),
#     pl.col("stage").alias("stage_list")
# ]).with_columns([
#     pl.col("stage_list").list.contains("Assessment").alias("hit_test"),
#     pl.col("stage_list").list.contains("Interview").alias("hit_interview"),
#     pl.col("stage_list").list.contains("Explicit Rejection").alias("hit_rej"),
#     (pl.lit(today) - pl.col("last_update")).dt.total_days().alias("days_ago")
# ])

# # Step C: outcome Logic
# job_funnel = job_funnel.with_columns(
#     final_status = pl.when(pl.col("hit_rej")).then(pl.lit("Rejection"))
#     .when(pl.col("days_ago") > 21).then(pl.lit("Ghosted"))
#     .otherwise(pl.lit("Active"))
# )

# # PRINT THIS to verify your interview count before plotting
# print(f"Verified Interview Count: {job_funnel.filter(pl.col('hit_interview')).height}")

In [ ]:
# import polars as pl
# import plotly.graph_objects as go
# from datetime import datetime


# today = datetime(2026, 5, 4)

# unique_stages = all_events.with_columns([
#     pl.col("stage").cast(pl.String),
#     pl.col("company_clean").cast(pl.String).fill_null("unknown").str.to_lowercase().str.strip_chars()
# ]).filter(
#     ~pl.col("company_clean").is_in(["", "none", "unknown", "null"])
# ).group_by(["company_clean", "role_clean", "stage"]).agg(
#     pl.col("event_date").min().alias("stage_date")
# )

# job_funnel = unique_stages.group_by(["company_clean", "role_clean"]).agg([
#     pl.col("stage_date").max().alias("last_update"),
#     pl.col("stage").alias("stage_list")
# ]).with_columns([
#     pl.col("stage_list").list.contains("Assessment").alias("hit_test"),
#     pl.col("stage_list").list.contains("Interview").alias("hit_interview"),
#     pl.col("stage_list").list.contains("Explicit Rejection").alias("hit_rej"),
#     (pl.lit(today) - pl.col("last_update")).dt.total_days().alias("days_ago")
# ])

# job_funnel = job_funnel.with_columns(
#     final_status = pl.when(pl.col("hit_rej")).then(pl.lit("Rejection"))
#     .when(pl.col("days_ago") > 21).then(pl.lit("Ghosted"))
#     .otherwise(pl.lit("Active"))
# )

# # 2. CALCULATE VOLUMES FOR LABELS & LINKS
# n_total = job_funnel.height
# n_test = job_funnel.filter(pl.col("hit_test")).height
# n_int = job_funnel.filter(pl.col("hit_interview")).height
# n_gho = job_funnel.filter(pl.col("final_status") == "Ghosted").height
# n_act = job_funnel.filter(pl.col("final_status") == "Active").height
# n_rej = job_funnel.filter(pl.col("final_status") == "Rejection").height

# # Splitting rejections to balance the flow
# rej_after_int = job_funnel.filter(pl.col("hit_interview") & pl.col("hit_rej")).height
# rej_after_test = job_funnel.filter(pl.col("hit_test") & ~pl.col("hit_interview") & pl.col("hit_rej")).height
# rej_initial = n_rej - rej_after_int - rej_after_test

# # 3. THE PROFESSIONAL SANKEY
# node_labels = [
#     f"Applied ({n_total})", f"Tests ({n_test})", f"Interviews ({n_int})",
#     f"Rejections ({n_rej})", f"Ghosted ({n_gho})", f"Active ({n_act})"
# ]

# # Slate/Steel Palette
# colors = ["#4682B4", "#9370DB", "#20B2AA", "#CD5C5C", "#DAA520", "#708090"]

# fig = go.Figure(data=[go.Sankey(
#     arrangement = "snap",
#     valueformat = "d",
#     node = dict(
#       pad = 100, 
#       thickness = 18,
#       label = node_labels,
#       color = colors,
#       hovertemplate = "%{label}<extra></extra>"
#     ),
#     link = dict(
#       source = [0, 1, 0, 1, 2, 0, 0], 
#       target = [1, 2, 3, 3, 3, 4, 5], 
#       value  = [n_test, n_int, rej_initial, rej_after_test, rej_after_int, n_gho, n_act],
#       color = "rgba(180, 180, 180, 0.25)"
#     )
# )])

# fig.update_layout(
#     title_text="<b>GRADUATE JOB SEARCH PIPELINE</b>",
#     font=dict(size=14, family="Arial"),
#     template="plotly_white",
#     height=800, width=1150
# )

# print(f"Plotting {n_int} Interviews.")
# fig.show()

In [ ]:

# today = datetime(2026, 5, 4)

# # Deduplicate stages per unique Job (Company + Role)
# unique_stages = all_events.with_columns([
#     pl.col("stage").cast(pl.String),
#     pl.col("company_clean").cast(pl.String).fill_null("unknown").str.to_lowercase().str.strip_chars(),
#     pl.col("role_clean").cast(pl.String).fill_null("unknown").str.to_lowercase().str.strip_chars()
# ]).group_by(["company_clean", "role_clean", "stage"]).agg(
#     pl.col("event_date").min().alias("stage_date")
# )

# # Aggregate to Job Level
# job_funnel = unique_stages.group_by(["company_clean", "role_clean"]).agg([
#     pl.col("stage_date").max().alias("last_update"),
#     pl.col("stage").alias("stage_list")
# ]).with_columns([
#     pl.col("stage_list").list.contains("Assessment").alias("hit_test"),
#     pl.col("stage_list").list.contains("Interview").alias("hit_interview"),
#     pl.col("stage_list").list.contains("Explicit Rejection").alias("hit_rej"),
#     (pl.lit(today) - pl.col("last_update")).dt.total_days().alias("days_ago")
# ])

# # Outcome Logic
# job_funnel = job_funnel.with_columns(
#     final_status = pl.when(pl.col("hit_rej")).then(pl.lit("Rejection"))
#     .when(pl.col("days_ago") > 21).then(pl.lit("Ghosted"))
#     .otherwise(pl.lit("Active"))
# )

In [ ]:
# import plotly.graph_objects as go

# # 1. Calculate Exact Volumes
# n_total = job_funnel.height
# n_test = job_funnel.filter(pl.col("hit_test")).height
# n_int = job_funnel.filter(pl.col("hit_interview")).height
# n_gho = job_funnel.filter(pl.col("final_status") == "Ghosted").height
# n_act = job_funnel.filter(pl.col("final_status") == "Active").height
# n_rej = job_funnel.filter(pl.col("final_status") == "Rejection").height

# # 2. Logic for Rejection splits (Math must balance)
# rej_after_int = job_funnel.filter(pl.col("hit_interview") & pl.col("hit_rej")).height
# rej_after_test = job_funnel.filter(pl.col("hit_test") & ~pl.col("hit_interview") & pl.col("hit_rej")).height
# rej_initial = n_rej - rej_after_int - rej_after_test

# # 3. Create Labels WITH counts (The fix for your "Names" issue)
# node_labels = [
#     f"Applied ({n_total})",
#     f"Tests ({n_test})",
#     f"Interviews ({n_int})",
#     f"Rejections ({n_rej})",
#     f"Ghosted ({n_gho})",
#     f"Active ({n_act})"
# ]

# # Professional Slate Palette
# colors = ["#4682B4", "#9370DB", "#20B2AA", "#CD5C5C", "#DAA520", "#708090"]

# fig = go.Figure(data=[go.Sankey(
#     arrangement = "snap",
#     valueformat = "d",
#     node = dict(
#       pad = 80, 
#       thickness = 15,
#       label = node_labels,
#       color = colors,
#       hovertemplate = "%{label}<extra></extra>"
#     ),
#     link = dict(
#       source = [0, 1, 0, 1, 2, 0, 0], 
#       target = [1, 2, 3, 3, 3, 4, 5], 
#       value  = [n_test, n_int, rej_initial, rej_after_test, rej_after_int, n_gho, n_act],
#       color = "rgba(180, 180, 180, 0.2)",
#       hovertemplate = "Flow: %{value} jobs<extra></extra>"
#     )
# )])

# fig.update_layout(
#     title_text="<b>GRADUATE JOB SEARCH PIPELINE</b>",
#     font=dict(size=14, family="Arial"),
#     template="plotly_white",
#     height=850, 
#     width=1150
# )

# fig.show()

In [ ]:
breakpoint()

In [ ]:

from datetime import datetime

today = datetime(2026, 5, 4)

# Deduplicate stages per job (Fixes the overcounting)
unique_stages = all_events.with_columns(
    pl.col("stage").cast(pl.String)
).group_by(["company_clean", "role_clean", "stage"]).agg(
    pl.col("event_date").min().alias("stage_date")
)

# Aggregate to Job Level
job_funnel = unique_stages.group_by(["company_clean", "role_clean"]).agg([
    pl.col("stage_date").max().alias("last_update"),
    pl.col("stage").alias("stage_list")
]).with_columns([
    pl.col("stage_list").list.contains("Assessment").alias("hit_test"),
    pl.col("stage_list").list.contains("Interview").alias("hit_interview"),
    pl.col("stage_list").list.contains("Explicit Rejection").alias("hit_rej"),
    (pl.lit(today) - pl.col("last_update")).dt.total_days().alias("days_ago")
])

# Define the Final Resting Places (Simplified)
job_funnel = job_funnel.with_columns(
    final_status = pl.when(pl.col("hit_rej")).then(pl.lit("Rejection"))
    .when(pl.col("days_ago") > 21).then(pl.lit("Ghosted"))
    .otherwise(pl.lit("Active"))
)

In [ ]:
import plotly.graph_objects as go

# 1. Path Volumes (Direct Flow)
total_apps = job_funnel.height
tests = job_funnel.filter(pl.col("hit_test")).height
interviews = job_funnel.filter(pl.col("hit_interview")).height

# 2. Terminal Node Volumes (Exit Points)
ghosted_total = job_funnel.filter(pl.col("final_status") == "Ghosted").height
active_total = job_funnel.filter(pl.col("final_status") == "Active").height
rejected_total = job_funnel.filter(pl.col("final_status") == "Rejection").height

# 3. Balanced Exit Logic (Where do rejections come from?)
# To keep the lines pretty, we distribute outcomes by the furthest stage reached
rej_after_interview = job_funnel.filter(pl.col("hit_interview") & pl.col("hit_rej")).height
rej_after_test = job_funnel.filter(pl.col("hit_test") & ~pl.col("hit_interview") & pl.col("hit_rej")).height
rej_at_screening = rejected_total - rej_after_interview - rej_after_test

labels = ["Total Applications", "Assessments", "Interviews", "Rejections", "Ghosted", "Active"]

fig = go.Figure(data=[go.Sankey(
    arrangement = "freeform",
    valueformat = ".0f", # MAKES NUMBERS VISIBLE
    node = dict(
      pad = 100, 
      thickness = 25,
      label = labels,
      color = ["#1f77b4", "#ab63fa", "#2ca02c", "#ef553b", "#ff7f0e", "#7f7f7f"]
    ),
    link = dict(
      # source -> target
      source = [0, 1, 0, 1, 2, 0, 0], 
      target = [1, 2, 3, 3, 3, 4, 5], 
      value  = [
          tests,                  # App -> Test
          interviews,             # Test -> Interview
          rej_at_screening,       # App -> Rejection
          rej_after_test,         # Test -> Rejection
          rej_after_interview,    # Interview -> Rejection
          ghosted_total,          # App -> Ghosted (Directly from start)
          active_total            # App -> Active (Current pipeline)
      ],
      color = "rgba(200, 200, 200, 0.4)"
  ))])

fig.update_layout(
    title_text="Job Application Conversion Funnel",
    height=900, width=1200, font_size=14
)
fig.show()

In [ ]:
# SGT (Surgical Ghosting/Interview Tool)
def manual_fix(df, company_name, has_int=None, is_rej=None):
    return df.with_columns(
        hit_interview = pl.when(pl.col("company_clean") == company_name.lower())
                        .then(has_int if has_int is not None else pl.col("hit_interview"))
                        .otherwise(pl.col("hit_interview")),
        hit_rej = pl.when(pl.col("company_clean") == company_name.lower())
                  .then(is_rej if is_rej is not None else pl.col("hit_rej"))
                  .otherwise(pl.col("hit_rej"))
    )

# Use it like this:
# job_funnel = manual_fix(job_funnel, "huawei", has_int=True, is_rej=False)
# job_funnel = manual_fix(job_funnel, "microsoft", has_int=True, is_rej=True)

In [ ]:
breakpoint()

In [ ]:
# import polars as pl
# from datetime import datetime

# today = datetime(2026, 5, 4)

# # Step A: Deduplicate and cast to String
# unique_stages = all_events.with_columns(
#     pl.col("stage").cast(pl.String) 
# ).group_by(["company_clean", "role_clean", "stage"]).agg(
#     pl.col("event_date").min().alias("stage_date")
# )

# # Step B: Aggregate to lists FIRST, then run logic
# job_funnel = (
#     unique_stages.group_by(["company_clean", "role_clean"])
#     .agg([
#         pl.col("stage_date").max().alias("last_update"),
#         pl.col("stage").alias("stage_list") # Create the list first
#     ])
#     .with_columns([
#         # Now we check the list for our milestones
#         pl.col("stage_list").list.contains("Assessment").alias("hit_test"),
#         pl.col("stage_list").list.contains("Interview").alias("hit_interview"),
#         pl.col("stage_list").list.contains("Explicit Rejection").alias("hit_rejection"),
#         (pl.lit(today) - pl.col("last_update")).dt.total_days().alias("days_since_update")
#     ])
# )

# # Step C: Assign Final Status
# job_funnel = job_funnel.with_columns(
#     final_status = pl.when(pl.col("hit_rejection")).then(
#         pl.when(pl.col("hit_interview")).then(pl.lit("Interview Rejection"))
#         .when(pl.col("hit_test")).then(pl.lit("Failed Assessment"))
#         .otherwise(pl.lit("Screening Rejection"))
#     )
#     .when(pl.col("hit_interview")).then(pl.lit("Active Interview / Offer"))
#     .when(pl.col("hit_test")).then(pl.lit("Assessment Pending"))
#     .when(pl.col("days_since_update") > 21).then(pl.lit("Ghosted >3 Weeks"))
#     .otherwise(pl.lit("Active / Pending"))
# )

In [ ]:
# import polars as pl
# from datetime import datetime

# today = datetime(2026, 5, 4)

# # ---------------------------------------------------------
# # Step 1: Data Cleaning & Mandatory Grouping (Fix the counts)
# # ---------------------------------------------------------

# # Start with your all_events dataframe and make it string-safe
# all_events_safe = all_events.with_columns(
#     pl.col("stage").cast(pl.String),
#     pl.col("company_clean").cast(pl.String).fill_null("").str.strip_chars().str.to_lowercase(),
#     pl.col("role_clean").cast(pl.String).fill_null("").str.strip_chars().str.to_lowercase()
# )

# # Mandatory cleaning filter based on your observed data issues
# # Drop entries with missing company/role before plotting
# cleaned_events = all_events_safe.filter(
#     (pl.col("company_clean") != "") & 
#     (pl.col("company_clean") != "none") &
#     (pl.col("role_clean") != "") &
#     (pl.col("role_clean") != "none")
# )

# # THE CRITICAL STEP: Collapse all duplicates by Company/Role/Stage
# # (e.g., all 4 Huawei interviews are binned here on the date of the first invite)
# unique_stages = cleaned_events.group_by(["company_clean", "role_clean", "stage"]).agg(
#     pl.col("event_date").min().alias("stage_date") # Earliest event triggers the milestone
# )

# # ---------------------------------------------------------
# # Step 2: Calculate the 3-Week rule (unchanged logic)
# # ---------------------------------------------------------
# job_funnel = (
#     unique_stages.group_by(["company_clean", "role_clean"])
#     .agg([
#         pl.col("stage_date").max().alias("last_update"),
#         pl.col("stage").alias("stage_list")
#     ])
#     .with_columns([
#         pl.col("stage_list").list.contains("Assessment").alias("hit_test"),
#         pl.col("stage_list").list.contains("Interview").alias("hit_interview"),
#         pl.col("stage_list").list.contains("Explicit Rejection").alias("hit_rejection"),
#         (pl.lit(today) - pl.col("last_update")).dt.total_days().alias("days_since_update")
#     ])
# )

In [ ]:
import polars as pl
from datetime import datetime

today = datetime(2026, 5, 4)

# Step A: Deduplicate and Clean Names
unique_stages = all_events.with_columns([
    pl.col("stage").cast(pl.String),
    pl.col("company_clean").fill_null("").str.to_lowercase().str.strip_chars()
]).filter(
    (pl.col("company_clean") != "") & (pl.col("company_clean") != "none")
).group_by(["company_clean", "role_clean", "stage"]).agg(
    pl.col("event_date").min().alias("stage_date")
)

# Step B: Build the Job-Level Funnel
job_funnel = unique_stages.group_by(["company_clean", "role_clean"]).agg([
    pl.col("stage_date").max().alias("last_update"),
    pl.col("stage").alias("stage_list")
]).with_columns([
    pl.col("stage_list").list.contains("Assessment").alias("hit_test"),
    pl.col("stage_list").list.contains("Interview").alias("hit_interview"),
    pl.col("stage_list").list.contains("Explicit Rejection").alias("hit_rejection"),
    (pl.lit(today) - pl.col("last_update")).dt.total_days().alias("days_since_update")
])

# Step C: Final Outcome Logic (3-Week Rule)
job_funnel = job_funnel.with_columns(
    outcome = pl.when(pl.col("hit_rejection")).then(pl.lit("Rejection"))
    .when(pl.col("days_since_update") > 21).then(pl.lit("Ghosted"))
    .otherwise(pl.lit("Active"))
)

In [ ]:
# Calculate Link Volumes
# Path 1: App -> Assessment
app_to_test = job_funnel.filter(pl.col("hit_test")).height
# Path 2: App -> Direct to Interview (Skipped Assessment)
app_to_int = job_funnel.filter(pl.col("hit_interview") & ~pl.col("hit_test")).height
# Path 3: App -> Early Rejection/Ghosting
app_to_rej = job_funnel.filter(~pl.col("hit_test") & ~pl.col("hit_interview") & (pl.col("outcome") == "Rejection")).height
app_to_gho = job_funnel.filter(~pl.col("hit_test") & ~pl.col("hit_interview") & (pl.col("outcome") == "Ghosted")).height
app_to_act = job_funnel.filter(~pl.col("hit_test") & ~pl.col("hit_interview") & (pl.col("outcome") == "Active")).height

# Path 4: Assessment -> Interview
test_to_int = job_funnel.filter(pl.col("hit_test") & pl.col("hit_interview")).height
# Path 5: Assessment -> Rejection/Ghosting
test_to_rej = job_funnel.filter(pl.col("hit_test") & ~pl.col("hit_interview") & (pl.col("outcome") == "Rejection")).height
test_to_gho = job_funnel.filter(pl.col("hit_test") & ~pl.col("hit_interview") & (pl.col("outcome") == "Ghosted")).height

# Path 6: Interview -> Final Outcomes
int_to_rej = job_funnel.filter(pl.col("hit_interview") & (pl.col("outcome") == "Rejection")).height
int_to_act = job_funnel.filter(pl.col("hit_interview") & (pl.col("outcome") == "Active")).height

In [ ]:
import polars as pl
from datetime import datetime

today = datetime(2026, 5, 4)

# Step A: Deduplicate - 4 Huawei emails = 1 Interview Milestone
unique_stages = all_events.with_columns([
    pl.col("stage").cast(pl.String),
    pl.col("company_clean").fill_null("").str.to_lowercase().str.strip_chars()
]).filter(
    (pl.col("company_clean") != "") & (pl.col("company_clean") != "none")
).group_by(["company_clean", "role_clean", "stage"]).agg(
    pl.col("event_date").min().alias("stage_date")
)

# Step B: Build Job-Level Summary
job_funnel = unique_stages.group_by(["company_clean", "role_clean"]).agg([
    pl.col("stage_date").max().alias("last_update"),
    pl.col("stage").alias("stage_list")
]).with_columns([
    pl.col("stage_list").list.contains("Assessment").alias("hit_test"),
    pl.col("stage_list").list.contains("Interview").alias("hit_interview"),
    pl.col("stage_list").list.contains("Explicit Rejection").alias("hit_rejection"),
    (pl.lit(today) - pl.col("last_update")).dt.total_days().alias("days_since_update")
])

# Step C: Terminal Outcome (Exclusive categories)
job_funnel = job_funnel.with_columns(
    outcome = pl.when(pl.col("hit_rejection")).then(pl.lit("Rejected"))
    .when(pl.col("days_since_update") > 21).then(pl.lit("Ghosted"))
    .otherwise(pl.lit("Still Active"))
)

In [ ]:
import plotly.graph_objects as go

# Calculate STRICT Path Volumes
# 1. Start to Assessment
to_test = job_funnel.filter(pl.col("hit_test")).height
# 2. Start to Direct Interview (No test)
to_int_direct = job_funnel.filter(pl.col("hit_interview") & ~pl.col("hit_test")).height
# 3. Assessment to Interview
test_to_int = job_funnel.filter(pl.col("hit_test") & pl.col("hit_interview")).height
# 4. Final Drops (Where the process ended for that job)
test_stop = job_funnel.filter(pl.col("hit_test") & ~pl.col("hit_interview")).height
int_stop = job_funnel.filter(pl.col("hit_interview")).height
start_stop = job_funnel.filter(~pl.col("hit_test") & ~pl.col("hit_interview")).height

# Mapping for terminal states
def get_counts(df_subset):
    return [
        df_subset.filter(pl.col("outcome") == "Rejected").height,
        df_subset.filter(pl.col("outcome") == "Ghosted").height,
        df_subset.filter(pl.col("outcome") == "Still Active").height
    ]

# terminal flows: [Rejected, Ghosted, Active]
flow_from_start = get_counts(job_funnel.filter(~pl.col("hit_test") & ~pl.col("hit_interview")))
flow_from_test = get_counts(job_funnel.filter(pl.col("hit_test") & ~pl.col("hit_interview")))
flow_from_int = get_counts(job_funnel.filter(pl.col("hit_interview")))

labels = ["Applied", "Assessment", "Interview", "Rejected", "Ghosted", "Still Active"]
# Indices: 0:App, 1:Test, 2:Int, 3:Rej, 4:Gho, 5:Active

fig = go.Figure(data=[go.Sankey(
    valueformat = ".6f",
    valuesuffix = " jobs",
    node = dict(
      pad = 50, thickness = 20,
      label = labels,
      color = ["#1f77b4", "#ab63fa", "#2ca02c", "#d62728", "#ff7f0e", "#7f7f7f"]
    ),
    link = dict(
      source = [0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2],
      target = [1, 2, 3, 4, 5, 2, 3, 4, 5, 3, 4, 5],
      value  = [to_test, to_int_direct, *flow_from_start, test_to_int, *flow_from_test, *flow_from_int]
  ))])

fig.update_layout(
    title="Deduplicated Job Pipeline (Terminal Outcomes)",
    height=800, width=1100, font_size=12
)
fig.show()

In [ ]:
# import plotly.graph_objects as go

# labels = ["Applied", "Assessment", "Interview", "Rejection", "Ghosted", "Active"]
# # Indices: 0:App, 1:Test, 2:Int, 3:Rej, 4:Gho, 5:Act

# fig = go.Figure(data=[go.Sankey(
#     arrangement = "freeform",
#     node = dict(
#       pad = 50, thickness = 20,
#       label = labels,
#       color = ["#1f77b4", "#ab63fa", "#2ca02c", "#d62728", "#ff7f0e", "#7f7f7f"]
#     ),
#     link = dict(
#       source = [0, 0, 0, 0, 0, 1, 1, 1, 2, 2],
#       target = [1, 2, 3, 4, 5, 2, 3, 4, 3, 5],
#       value  = [app_to_test, app_to_int, app_to_rej, app_to_gho, app_to_act, 
#                 test_to_int, test_to_rej, test_to_gho, int_to_rej, int_to_act]
#   ))])

# fig.update_layout(
#     title="Cleaned Job Funnel (Deduplicated)",
#     font_size=12,
#     height=800, # Increased height to stop the 'squawking'
#     width=1000
# )
# fig.show()

In [ ]:
import plotly.express as px

# Truncate to '1d' so you get the exact date pulse for that 3-month zoom
histo_df = unique_stages.with_columns(
    pl.col("stage_date").dt.truncate("1d").alias("event_day")
).to_pandas()

fig = px.histogram(
    histo_df,
    x="event_day",
    color="stage",
    title="Daily Application Activity Pulse (Grouped & Deduplicated Events)",
    barmode="stack", # Use 'group' to see the pulse side-by-side
    color_discrete_map={
        "Applied": "#1f77b4", # Blue
        "Assessment": "#ab63fa",   # Purple
        "Interview": "#2ca02c",  # Green
        "Explicit Rejection": "#ef553b"      # Red
    },
    category_orders={"stage": ["Applied", "Assessment", "Interview", "Explicit Rejection"]},
)

fig.update_layout(
    xaxis_title="Timeline",
    yaxis_title="Count of Distinct Events",
    bargap=0.1,
    height=500
)

# Enable the interactive Range Slider to choose the date range
fig.update_xaxes(rangeslider_visible=True, type="date")
fig.show()

In [ ]:
import plotly.express as px

# Truncate to day for maximum granularity
histo_df = unique_stages.with_columns(
    pl.col("stage_date").dt.truncate("1d").alias("event_day")
).to_pandas()

fig = px.histogram(
    histo_df,
    x="event_day",
    color="stage",
    title="Daily Application Activity (True Event Dates)",
    barmode="stack",
    category_orders={"stage": ["Applied", "Assessment", "Interview", "Explicit Rejection"]},
    color_discrete_map={
        "Applied": "#1f77b4", 
        "Assessment": "#ab63fa", 
        "Interview": "#2ca02c", 
        "Explicit Rejection": "#ef553b"
    }
)

fig.update_layout(
    xaxis_title="Date of Event",
    yaxis_title="Volume",
    bargap=0.1,
    height=500
)

# Enable the range slider for choosing the date range
fig.update_xaxes(rangeslider_visible=True, type="date")
fig.show()

In [ ]:
import plotly.graph_objects as go

# Calculate transition volumes from our new job_funnel
total = job_funnel.height
tests = job_funnel.filter(pl.col("hit_test")).height
interviews = job_funnel.filter(pl.col("hit_interview")).height
ghosted = job_funnel.filter(pl.col("final_status") == "Ghosted >3 Weeks").height
rejected = job_funnel.filter(pl.col("hit_rejection")).height
active = total - rejected - ghosted

labels = ["Total Applications", "Assessments", "Interviews", "Rejections", "Ghosted", "Active"]

fig = go.Figure(data=[go.Sankey(
    arrangement ="snap",
    node = dict(
      pad = 100, # Massive padding stops the overlapping text/boxes
      thickness = 20,
      label = labels,
      color = ["#1f77b4", "#ab63fa", "#2ca02c", "#ef553b", "#ff7f0e", "#7f7f7f"]
    ),
    link = dict(
      source = [0, 1, 0, 1, 2, 0, 0], # From Node
      target = [1, 2, 3, 3, 3, 4, 5], # To Node
      # Note: value math ensures the flow balances correctly
      value  = [tests, interviews, (rejected - (tests+interviews)), (tests-interviews), 1, ghosted, active] 
  ))])

fig.update_layout(
    title="Application Process Flow",
    height=900, # High vertical space
    width=1200,
    font_size=14
)
fig.show()

In [ ]:
import polars as pl

# Collapse duplicates: For every company/role, find the FIRST time you hit a stage
unique_events = all_events.group_by(["company_clean", "role_clean", "stage"]).agg(
    pl.col("event_date").min().alias("event_date")
)

# Now, recreate the histogram timeline with clean data
apps = unique_events.filter(pl.col("stage") == "Applied").select([
    pl.col("event_date").alias("date"),
    pl.lit("Application Submitted").alias("event_type")
])

tests = unique_events.filter(pl.col("stage") == "Assessment").select([
    pl.col("event_date").alias("date"),
    pl.lit("Assessment Received").alias("event_type")
])

interviews = unique_events.filter(pl.col("stage") == "Interview").select([
    pl.col("event_date").alias("date"),
    pl.lit("Interview Invitation").alias("event_type")
])

rejections = unique_events.filter(pl.col("stage") == "Explicit Rejection").select([
    pl.col("event_date").alias("date"),
    pl.lit("Explicit Rejection").alias("event_type")
])

timeline_df = pl.concat([apps, tests, interviews, rejections])

In [ ]:
import plotly.express as px

# Truncate to Day ('1d') or Week ('1w') as requested
plot_df = timeline_df.with_columns(
    pl.col("date").dt.truncate("1d").alias("period")
).to_pandas()

fig = px.histogram(
    plot_df,
    x="period",
    color="event_type",
    title="Daily Job Hunt Pulse (Grouped Milestones)",
    # Use barmode='group' to see the bars side-by-side or 'stack' for total volume
    barmode='stack',
    color_discrete_map={
        "Application Submitted": "#1f77b4", 
        "Assessment Received": "#ab63fa",   
        "Interview Invitation": "#2ca02c",  
        "Explicit Rejection": "#ef553b"      
    }
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Count of Events",
    height=500
)

# Enable the Range Slider for date selection
fig.update_xaxes(rangeslider_visible=True, type="date")
fig.show()

In [ ]:
import plotly.graph_objects as go


fig = go.Figure(data=[go.Sankey( # Forces nodes into clean columns
    node = dict(
      pad = 50,         # Large vertical padding to stop overlaps
      thickness = 25,
      line = dict(color = "black", width = 0.5),
      label = unique_nodes,
      color = node_colors
    ),
    link = dict(
      source = source_idx,
      target = target_idx,
      value = values,
      color = link_colors
  ))])

fig.update_layout(
    title_text="Job Application Funnel (Grouped Stages)",
    height=900,  # Tall enough to handle many nodes
    width=1100, 
    font_size=14 # Larger font for better readability
)
fig.show()

In [ ]:
# outer_join_expr = """
# WITH RankedMails AS (
#     -- Step 1: Rank emails per job from newest (1) to oldest
#     SELECT 
#         company_clean AS m_company, 
#         role_clean AS m_role, 
#         update_category,
#         ROW_NUMBER() OVER(
#             PARTITION BY company_clean, role_clean 
#             ORDER BY date DESC
#         ) AS rn
#     FROM mail
# ),
# MailGroups AS (
#     -- Step 2: Get the counts and earliest application date
#     SELECT 
#         company_clean AS m_company, 
#         role_clean AS m_role, 
#         COUNT(DISTINCT update_category) AS distinct_roles, 
#         COUNT(*) AS mail_count,
#         MIN(date) AS mail_date
#     FROM mail
#     GROUP BY company_clean, role_clean
# ),
# LinkedInGroups AS (
#     -- Step 3: Get the LinkedIn data
#     SELECT 
#         company_clean AS l_company, 
#         role_clean AS l_role, 
#         COUNT(*) AS linkedin_count,
#         MIN(applied_date) AS linkedin_date
#     FROM linkedin2
#     GROUP BY company_clean, role_clean
# )
# SELECT 
#     -- Explicitly combine the names so nothing is ever NULL
#     COALESCE(m.m_company, l.l_company) AS company_clean, 
#     COALESCE(m.m_role, l.l_role) AS role_clean, 
    
#     -- Step 4: Grab the chronological status where rn = 1
#     COALESCE(r.update_category, 'Pending / No Email') AS application_status,
    
#     COALESCE(m.distinct_roles, 0) AS distinct_roles, 
#     COALESCE(m.mail_count, 0) AS mail_count, 
#     COALESCE(l.linkedin_count, 0) AS linkedin_count,
    
#     -- The Date Resolver
#     CASE
#         WHEN m.mail_date IS NULL THEN l.linkedin_date
#         WHEN l.linkedin_date IS NULL THEN m.mail_date
#         WHEN m.mail_date < l.linkedin_date THEN m.mail_date
#         ELSE l.linkedin_date
#     END AS first_applied_date
    
# FROM MailGroups m
# -- Join the ranked emails to attach the newest status to the mail groups
# LEFT JOIN RankedMails r 
#     ON m.m_company = r.m_company 
#     AND m.m_role = r.m_role 
#     AND r.rn = 1
# -- Now do the full outer join with LinkedIn
# FULL OUTER JOIN LinkedInGroups l 
#     ON m.m_company = l.l_company 
#     AND m.m_role = l.l_role
# ORDER BY first_applied_date DESC
# """

# # Execute the query
# all_applications = con.sql(outer_join_expr)

# # View the perfectly matched, chronologically accurate DataFrame
# all_applications

In [ ]:
# sns.histplot(all_applications['distinct_roles'], x=all_applications['company_clean'])

In [ ]:

# ALTERNATE



# # 1. Create the aggregated summaries
# mail_summary = (
#     mail_fetch
#     .group_by(["company_clean", "role_clean"])
#     .aggregate(
#         distinct_roles=mail_fetch.update_category.nunique(),
#         mail_count=mail_fetch.count()
#     )
# )

# li2_summary = (
#     li2_fetch
#     .group_by(["company_clean", "role_clean"])
#     .aggregate(linkedin_count=li2_fetch.count())
# )

# # 2. The FULL OUTER JOIN
# # This ensures EVERY application from both tables survives
# all_applications = mail_summary.outer_join(
#     li2_summary, 
#     ["company_clean", "role_clean"]
# )

# # 3. Clean up the missing counts
# # If an app was only in LinkedIn, mail_count becomes 0 instead of NULL
# all_applications = all_applications.mutate(
#     mail_count=ibis.coalesce(all_applications.mail_count, 0),
#     linkedin_count=ibis.coalesce(all_applications.linkedin_count, 0),
#     distinct_roles=ibis.coalesce(all_applications.distinct_roles, 0)
# )

# # Sort it to see your most active applications first
# all_applications = all_applications.order_by(ibis.desc("mail_count"))

# # Execute and view
# all_applications_df = all_applications.execute()

In [ ]:
# #ALTERNATE 2
# # 1. Add MIN dates to the summaries
# mail_summary = (
#     mail_fetch
#     .group_by(["company_clean", "role_clean"])
#     .aggregate(
#         distinct_roles=mail_fetch.update_category.nunique(),
#         mail_count=mail_fetch.count(),
#         mail_date=mail_fetch.date.min()
#     )
# )

# li2_summary = (
#     li2_fetch
#     .group_by(["company_clean", "role_clean"])
#     .aggregate(
#         linkedin_count=li2_fetch.count(),
#         linkedin_date=li2_fetch.applied_date.min()
#     )
# )

# # 2. Join them
# all_applications = mail_summary.outer_join(
#     li2_summary, 
#     ["company_clean", "role_clean"]
# )

# # 3. Join them using explicit conditions so we can control the columns
# all_applications = mail_summary.outer_join(
#     li2_summary, 
#     (mail_summary.company_clean == li2_summary.company_clean) & 
#     (mail_summary.role_clean == li2_summary.role_clean)
# )

# # 4. Clean up EVERYTHING (Rescue the names, fix the math, fix the dates)
# all_applications = all_applications.mutate(
#     # Rescue the company and role names!
#     clean_company=ibis.coalesce(mail_summary.company_clean, li2_summary.company_clean),
#     clean_role=ibis.coalesce(mail_summary.role_clean, li2_summary.role_clean),
    
#     # Fix the math
#     mail_count=ibis.coalesce(all_applications.mail_count, 0),
#     linkedin_count=ibis.coalesce(all_applications.linkedin_count, 0),
#     distinct_roles=ibis.coalesce(all_applications.distinct_roles, 0),
    
#     # Fix the date
#     first_applied_date=ibis.case()
#         .when(mail_summary.mail_date.isnull(), li2_summary.linkedin_date)
#         .when(li2_summary.linkedin_date.isnull(), mail_summary.mail_date)
#         .when(mail_summary.mail_date < li2_summary.linkedin_date, mail_summary.mail_date)
#         .else_(li2_summary.linkedin_date)
#         .end()
# )

# # 5. Select only the clean columns to drop the duplicates/nulls, and sort
# all_applications = (
#     all_applications
#     .select(
#         company_clean=all_applications.clean_company,
#         role_clean=all_applications.clean_role,
#         distinct_roles=all_applications.distinct_roles,
#         mail_count=all_applications.mail_count,
#         linkedin_count=all_applications.linkedin_count,
#         first_applied_date=all_applications.first_applied_date
#     )
#     .order_by(ibis.desc("first_applied_date"))
# )

# all_applications_df = all_applications.execute()
# all_applications_df

import plotly.graph_objects as go

# (Assuming you've run the 'transitions' logic from the previous turn with the unique_events)
# unique_nodes = ["Applied", "Assessment", "Interview", "Explicit Rejection", "Ghosted / Assumed Rejection", "Active / Pending"]

fig = go.Figure(data=[go.Sankey(
    arrangement = "perpendicular", # Forces nodes into clean columns
    node = dict(
      pad = 50,         # Large vertical padding to stop overlaps
      thickness = 25,
      line = dict(color = "black", width = 0.5),
      label = unique_nodes,
      color = node_colors
    ),
    link = dict(
      source = source_idx,
      target = target_idx,
      value = values,
      color = link_colors
  ))])

fig.update_layout(
    title_text="Job Application Funnel (Grouped Stages)",
    height=900,  # Tall enough to handle many nodes
    width=1100, 
    font_size=14 # Larger font for better readability
)
fig.show()

In [ ]:
import polars as pl
import plotly.express as px
from datetime import datetime

# 1. Start with the raw SQL output and set today's date
df = all_applications.to_polars()
today = datetime(2026, 5, 4)

# 2. Build the 'final_outcome' category column
df = df.with_columns(
    (pl.lit(today) - pl.col("first_applied_date")).dt.total_days().alias("days_since_app")
).with_columns(
    final_outcome=pl.when(pl.col("has_interview") == 1).then(
        pl.when(pl.col("is_rejected") == 1).then(pl.lit("Interview Rejection"))
        .otherwise(pl.lit("Active Interview / Offer"))
    )
    .when(pl.col("has_test") == 1).then(
        pl.when(pl.col("is_rejected") == 1).then(pl.lit("Failed Assessment"))
        .otherwise(pl.lit("Assessment Pending"))
    )
    .when(pl.col("is_rejected") == 1).then(pl.lit("Screening Rejection"))
    .when(pl.col("days_since_app") > 21).then(pl.lit("Ghosted >3 Weeks"))
    .otherwise(pl.lit("Active / Pending"))
)

# 3. TRUNCATE TO WEEK OR DAY (This is the fix)
# Change "1w" to "1d" if you truly want it day-by-day
time_df = df.with_columns(
    pl.col("first_applied_date").dt.truncate("1d").alias("apply_period") 
).to_pandas() 

# 4. Plot using the new 'apply_period' column
fig = px.histogram(
    time_df,
    x="apply_period", # Updated to the new column
    color="final_outcome", 
    title="Application Outcomes Over Time",
    labels={"apply_period": "Date Applied", "count": "Number of Applications"},
    category_orders={
        "final_outcome": [
            "Active / Pending", "Assessment Pending", "Active Interview / Offer", 
            "Ghosted >3 Weeks", "Screening Rejection", "Failed Assessment", "Interview Rejection"
        ]
    }, 
    color_discrete_map={
        "Active / Pending": "#7f7f7f",           
        "Assessment Pending": "#aec7e8",         
        "Active Interview / Offer": "#2ca02c",   
        "Ghosted >3 Weeks": "#ffbb78",           
        "Screening Rejection": "#1f77b4",        
        "Failed Assessment": "#ff7f0e",          
        "Interview Rejection": "#d62728"         
    }
)

fig.update_layout(barmode='stack', xaxis_title="Date", yaxis_title="Applications")

# Keep the interactive range slider!
fig.update_xaxes(
    rangeslider_visible=True,
    type="date" 
)

fig.show()

In [ ]:
# import polars as pl

# # Let's say you want to manually update a "Data Scientist" application at "Standard Bank" 
# # because you know you got an interview, but the email parser missed it.

# df = df.with_columns(
#     # 1. Force 'has_interview' to 1 for this specific job
#     has_interview=pl.when(
#         (pl.col("company_clean") == "standard bank") & 
#         (pl.col("role_clean") == "data scientist")
#     )
#     .then(1) # Set to 1 (True)
#     .otherwise(pl.col("has_interview")), # Leave everyone else alone
    
#     # 2. Force 'is_rejected' to 0 (in case it was accidentally marked rejected)
#     is_rejected=pl.when(
#         (pl.col("company_clean") == "standard bank") & 
#         (pl.col("role_clean") == "data scientist")
#     )
#     .then(0) 
#     .otherwise(pl.col("is_rejected"))
# )

# # You can chain multiple conditions if you need to fix several companies at once:
# df = df.with_columns(
#     is_rejected=pl.when(pl.col("company_clean").is_in(["company_a", "company_b"]))
#     .then(1)
#     .otherwise(pl.col("is_rejected"))
# )

In [ ]:
import plotly.graph_objects as go
import polars as pl
from datetime import datetime

# 1. Convert to Polars and calculate the 3-week "Ghosting" rule
df = all_applications.to_polars()
today = datetime(2026, 5, 4)

df = df.with_columns(
    (pl.lit(today) - pl.col("first_applied_date")).dt.total_days().alias("days_since_app")
)

# 2. Calculate the exact volume for every specific path
# --- Initial Splits from 'Total Applications' ---
screen_reject_cnt = df.filter((pl.col("has_test") == 0) & (pl.col("has_interview") == 0) & (pl.col("is_rejected") == 1)).height
ghosted_cnt = df.filter((pl.col("has_test") == 0) & (pl.col("has_interview") == 0) & (pl.col("is_rejected") == 0) & (pl.col("days_since_app") > 21)).height
pending_cnt = df.filter((pl.col("has_test") == 0) & (pl.col("has_interview") == 0) & (pl.col("is_rejected") == 0) & (pl.col("days_since_app") <= 21)).height

taken_assessment_cnt = df.filter(pl.col("has_test") == 1).height
direct_to_interview_cnt = df.filter((pl.col("has_test") == 0) & (pl.col("has_interview") == 1)).height

# --- Splits from 'Taken Assessment' ---
failed_assess_cnt = df.filter((pl.col("has_test") == 1) & (pl.col("has_interview") == 0) & (pl.col("is_rejected") == 1)).height
pending_assess_cnt = df.filter((pl.col("has_test") == 1) & (pl.col("has_interview") == 0) & (pl.col("is_rejected") == 0)).height
assess_to_interview_cnt = df.filter((pl.col("has_test") == 1) & (pl.col("has_interview") == 1)).height

# --- Splits from 'Interview Stage' ---
interview_reject_cnt = df.filter((pl.col("has_interview") == 1) & (pl.col("is_rejected") == 1)).height
active_interview_cnt = df.filter((pl.col("has_interview") == 1) & (pl.col("is_rejected") == 0)).height

total_apps = df.height

# 3. Define the Sankey Nodes
nodes = [
    f"Total Applications ({total_apps})",     # 0
    f"Screening Rejection ({screen_reject_cnt})", # 1
    f"Ghosted >3 Weeks ({ghosted_cnt})",      # 2
    f"Active / Pending ({pending_cnt})",      # 3
    f"Taken Assessment ({taken_assessment_cnt})", # 4
    f"Failed Assessment ({failed_assess_cnt})",   # 5
    f"Assessment Pending ({pending_assess_cnt})", # 6
    f"Interview Stage ({direct_to_interview_cnt + assess_to_interview_cnt})", # 7 (Merged node)
    f"Interview Rejection ({interview_reject_cnt})", # 8
    f"Offer / Active Interview ({active_interview_cnt})" # 9
]

# 4. Map the exact Source-to-Target flow
source = [
    0, 0, 0, 0, 0,  # From Total Apps -> (Screen Reject, Ghosted, Pending, Assessment, Direct to Interview)
    4, 4, 4,        # From Assessment -> (Failed Assess, Pending Assess, Advanced to Interview)
    7, 7            # From Interview Stage -> (Interview Reject, Active Interview)
]

target = [
    1, 2, 3, 4, 7,  # Targets for Total Apps
    5, 6, 7,        # Targets for Taken Assessment
    8, 9            # Targets for Interview Stage
]

value = [
    screen_reject_cnt, ghosted_cnt, pending_cnt, taken_assessment_cnt, direct_to_interview_cnt,
    failed_assess_cnt, pending_assess_cnt, assess_to_interview_cnt,
    interview_reject_cnt, active_interview_cnt
]

fig = go.Figure(data=[go.Sankey(
    valueformat = ".0f",
    node = dict(
      pad = 30, # Increased from 15: Forces more vertical space between nodes
      thickness = 20,
      line = dict(color = "black", width = 0.5),
      label = nodes,
      color = ["#1f77b4", "#d62728", "#7f7f7f", "#aec7e8", "#ff7f0e", "#d62728", "#ffbb78", "#2ca02c", "#d62728", "#98df8a"]
    ),
    link = dict(
      source = source,
      target = target,
      value = value,
      color = "rgba(200, 200, 200, 0.4)"
  ))])

fig.update_layout(
    title_text="Accurate Job Application Flow", 
    font_size=12,
    height=800, # <-- This is the magic fix. Increase to 1000 if still tight.
    width=1000  # <-- Gives the horizontal flow more room to stretch
)


fig.show()